# Solar: exact reproducible candidate
Jacek Adamek - proposed competition update, 16 October 2026. Nothing is sent to Kaggle.

Clone the complete repository. The three frozen reference weights are now included
under submission-update-2026-10-16/weights. Obtain competition inputs separately
through the organizer. Set SOLAR_DATA to MAGFiLO_1.0_Kaggle_2026.
Run this notebook from the repository root or this dated update directory.
Install requirements.txt in the active Python environment.
Set SOLAR_EXPORT_ALL=1 to run the entire 180-image export in the final code cell.


In [ ]:
import os, sys, json, hashlib, subprocess, importlib.util
from pathlib import Path
import numpy as np
from PIL import Image
from pycocotools import mask as coco
REPO = Path.cwd().resolve()
if not (REPO/'releases/2026-10-16').is_dir(): REPO = REPO.parent
RELEASE = Path(os.environ.get('SOLAR_RELEASE', str(REPO/'releases/2026-10-16'))).resolve()
PACKAGE = REPO/'submission-update-2026-10-16'
DATA = Path(os.environ.get('SOLAR_DATA', './MAGFiLO_1.0_Kaggle_2026')).resolve()
WEIGHTS = PACKAGE/'weights'
subprocess.run([sys.executable,str(PACKAGE/'verify_materials.py')],check=True,capture_output=True)
assert (RELEASE/'verify_release.py').is_file(), 'Configure SOLAR_RELEASE'
subprocess.run([sys.executable, str(RELEASE/'verify_release.py')], check=True, capture_output=True)
images = sorted((DATA/'test/test_images').glob('*.jp*g'))
assert len(images) == 180
print('Release integrity verified; 180 test images registered.')


## From pixels to baseline masks and refined instance labels
The imported frozen source computes the disk, 24 low-resolution pixel features,
pixel predictions, connected components, 14 component features and learned rejection.
The ridge stage computes ten full-resolution features and refines existing masks.
These calls read current pixels, not training annotations or probability caches.


In [ ]:
spec = importlib.util.spec_from_file_location('release_adapter', RELEASE/'predict.py')
adapter = importlib.util.module_from_spec(spec); spec.loader.exec_module(adapter)
(RELEASE/'work').mkdir(exist_ok=True)
sys.path.insert(0, str(RELEASE/'outputs/solar-ridge-v8'))
import image_only as ridge
adapter.install_weights(ridge.base, WEIGHTS)
with Image.open(images[0]) as image_file:
    image = np.asarray(image_file)
baseline = ridge.base.segment(image, 'rf_filter')
labels = ridge.segment(image)
assert labels.shape == (2048, 2048)
assert set(np.unique(labels)) == set(np.unique(baseline))
print('First test image:', images[0].name)
print('Baseline/refined instance count:', int(baseline.max()), int(labels.max()))
print('Refined label-array SHA256:', hashlib.sha256(labels.tobytes()).hexdigest())


In [ ]:
rows = []
for number in range(1, int(labels.max())+1):
    mask = labels == number
    rle = coco.encode(np.asfortranarray(mask.astype(np.uint8)))
    assert np.array_equal(coco.decode(rle).astype(bool), mask)
    rows.append({'filament_id': f'{images[0].stem}_{number}', 'segmentation_rle': rle['counts'].decode('ascii')})
print('Exact per-instance RLE round-trips:', len(rows))


## Entire test-set export
The accompanying predict_submission.py performs these frozen predictions on all 180 images,
checks every RLE round-trip and writes a CSV only after all image records are complete.
Use a new output directory. The exporter never contacts Kaggle.
Run from this notebook's directory after setting the paths above:

```text
python predict_submission.py --release PUBLIC_RELEASE_DIRECTORY --images AUTHORIZED_DATA/test/test_images --weights submission-update-2026-10-16/weights --output NEW_OUTPUT/candidate-ridge-v8.csv --workers 3
```

The final code cell can run this complete export and checks its exact expected CSV digest. Uploading is a separate future action.


## Train from authorized masks instead of using reference weights
The public REPRODUCE.md explains preparing a fresh workspace, then running
research.py rf, research.py tune --method rf, component_followup.py fit,
and solar-ridge-v8/train_evaluate.py. Do not modify frozen study outcomes.
Full training is not executed by this notebook. A newly fitted model must use
its own frozen protocols and measured outcomes rather than claiming historical serialized hashes.

## Interpretation
Combined reused-development PQ: 0.276155 to 0.277402; FP: 810 to 807;
thin annotation-level matches: 126/442 to 127/442. Confidence intervals include zero.
The additional 40 dates add no thin matches. No new Kaggle score is known.


In [ ]:
RUN_FULL_EXPORT = os.environ.get('SOLAR_EXPORT_ALL', '0') == '1'
if RUN_FULL_EXPORT:
    output = REPO/'work/public-reproduction/candidate-ridge-v8.csv'
    subprocess.run([sys.executable,str(PACKAGE/'predict_submission.py'),'--images',str(DATA/'test/test_images'),'--output',str(output),'--workers','3'],check=True)
    expected = json.loads((PACKAGE/'candidate-ridge-v8.audit.json').read_text())['csv_sha256']
    actual = hashlib.sha256(output.read_bytes()).hexdigest()
    assert actual == expected, (actual, expected)
    print('All 180 images reproduced; exact CSV SHA256:', actual)
else:
    print('Full export skipped. Set SOLAR_EXPORT_ALL=1 to execute all 180 images.')
